# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Shehzadi434/flyrank-Internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


**Top 10 Actions:**

| Rank | Action | Reason Code | Impressions | CTR |
|------|--------|-------------|-------------|-----|
| 1 | Refresh content | high_impressions_low_ctr | 617,124 | 0.009 |
| 2 | Refresh content | high_impressions_low_ctr | 245,276 | 0.006 |
| 3 | Refresh content | high_impressions_low_ctr | 221,310 | 0.003 |
| 4 | Refresh content | high_impressions_low_ctr | 203,497 | 0.001 |
| 5 | Refresh content | high_impressions_low_ctr | 186,983 | 0.003 |
| 6 | Monitor | other | 205,045 | 0.012 |
| 7 | Refresh content | high_impressions_low_ctr | 151,166 | 0.003 |
| 8 | Refresh content | high_impressions_low_ctr | 164,885 | 0.002 |
| 9 | Refresh content | high_impressions_low_ctr | 142,304 | 0.002 |
| 10 | Refresh content | high_impressions_low_ctr | 244,931 | 0.003 |

**Reason Code Distribution (n=176,738):**
- `other`: 116,668 (66.0%)
- `high_impressions_low_ctr`: 59,393 (33.6%)
- `position_opportunity`: 475 (0.3%)
- `stale_visible_page`: 202 (0.1%)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. Ready to Go.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. Ready to Go.


In [3]:
print("=" * 50)
print("RANKED ACTIONS + REASON CODES")
print("=" * 50)

# Build the baseline queue (same logic as ML-07)
import pandas as pd
import numpy as np
import os
import duckdb
from google.colab import userdata

cache_path = 'work/outputs/feature_vector_march2026.parquet'

# Load or build feature vector
if os.path.exists(cache_path):
    print("Loading cached feature vector...")
    feature_vector = pd.read_parquet(cache_path)
    print(f" Loaded: {len(feature_vector):,} rows")
else:
    print("Cache not found. Building feature vector from warehouse...")

    con = duckdb.connect()
    HF_TOKEN = userdata.get('HF_TOKEN')
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

    REL = 'hf://datasets/FlyRank/internship-warehouse'

    feature_vector = con.sql(f"""
        WITH daily_features AS (
            SELECT
                d.content_hash_id,
                d.client_hash_id,
                SUM(d.gsc_impressions) AS impressions_90d,
                SUM(d.gsc_clicks) AS clicks_90d,
                AVG(d.gsc_avg_position) AS avg_position_90d,
                CASE
                    WHEN SUM(d.gsc_impressions) > 0
                    THEN SUM(d.gsc_clicks) * 1.0 / SUM(d.gsc_impressions)
                    ELSE 0
                END AS ctr_90d,
                COUNT(DISTINCT d.report_date) AS days_active
            FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') d
            WHERE d.gsc_impressions IS NOT NULL AND d.gsc_impressions > 0
            GROUP BY d.content_hash_id, d.client_hash_id
        )
        SELECT
            d.*,
            DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days,
            c.content_type,
            c.word_count,
            c.search_volume,
            c.main_intent
        FROM daily_features d
        LEFT JOIN read_parquet('{REL}/dim_content.parquet') c
            ON d.content_hash_id = c.content_hash_id
        WHERE c.content_created_date IS NOT NULL
    """).df()

    os.makedirs('work/outputs', exist_ok=True)
    feature_vector.to_parquet(cache_path)
    print(f" Built and cached: {len(feature_vector):,} rows")

# Fill missing values
feature_vector['ctr_90d'] = feature_vector['ctr_90d'].fillna(0)
feature_vector['avg_position_90d'] = feature_vector['avg_position_90d'].fillna(10)
feature_vector['content_age_days'] = feature_vector['content_age_days'].fillna(0)

# Calculate baseline score (from ML-07)
feature_vector['baseline_score'] = (
    feature_vector['impressions_90d'] *
    (1 - feature_vector['ctr_90d']) *
    (feature_vector['content_age_days'] / 365)
)

# Assign reason codes
def assign_reason(row):
    if row['impressions_90d'] > 500 and row['ctr_90d'] < 0.01:
        return 'high_impressions_low_ctr'
    elif row['content_age_days'] > 365 and row['impressions_90d'] > 500:
        return 'stale_visible_page'
    elif row['avg_position_90d'] > 10 and row['impressions_90d'] > 500:
        return 'position_opportunity'
    else:
        return 'other'

feature_vector['reason_code'] = feature_vector.apply(assign_reason, axis=1)

# Create the queue
queue = feature_vector.sort_values('baseline_score', ascending=False)

# Select key columns
queue_output = queue[['content_hash_id', 'client_hash_id', 'baseline_score',
                      'reason_code', 'impressions_90d', 'ctr_90d',
                      'content_age_days', 'avg_position_90d']].copy()

# Save to CSV
os.makedirs('work/outputs', exist_ok=True)
queue_output.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f" Queue built and saved: {len(queue_output):,} rows")

print("\nTop 10 recommended actions:")
print("-" * 70)
print(f"{'Rank':<6} {'Action':<20} {'Reason Code':<25} {'Impressions':<12} {'CTR':<10}")
print("-" * 70)

actions = {
    'high_impressions_low_ctr': 'Refresh content',
    'stale_visible_page': 'Update content',
    'position_opportunity': 'Improve SEO',
    'other': 'Monitor'
}

for i, (idx, row) in enumerate(queue.head(10).iterrows(), 1):
    action = actions.get(row['reason_code'], 'Monitor')
    print(f"{i:<6} {action:<20} {row['reason_code']:<25} {row['impressions_90d']:<12.0f} {row['ctr_90d']:<10.4f}")

print("\nReason Code Distribution:")
print(queue['reason_code'].value_counts())

RANKED ACTIONS + REASON CODES
Cache not found. Building feature vector from warehouse...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 Built and cached: 176,738 rows
 Queue built and saved: 176,738 rows

Top 10 recommended actions:
----------------------------------------------------------------------
Rank   Action               Reason Code               Impressions  CTR       
----------------------------------------------------------------------
1      Refresh content      high_impressions_low_ctr  617124       0.0092    
2      Refresh content      high_impressions_low_ctr  245276       0.0060    
3      Refresh content      high_impressions_low_ctr  221310       0.0033    
4      Refresh content      high_impressions_low_ctr  203497       0.0014    
5      Refresh content      high_impressions_low_ctr  186983       0.0031    
6      Monitor              other                     205045       0.0119    
7      Refresh content      high_impressions_low_ctr  151166       0.0027    
8      Refresh content      high_impressions_low_ctr  164885       0.0024    
9      Refresh content      high_impressions_low_ctr  1423

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


**Intended Use:**
- **Who:** Content managers, SEO teams, editorial leads
- **What:** Prioritize pages for review and refresh
- **When:** Weekly review cycle
- **How:** Review top 50, verify reason codes, take action

**Limits:**
- ❌ Not a guarantee of recovery
- ❌ Not causal — based on observed patterns
- ❌ Not a replacement for human judgment
- ❌ Limited to observed signals (impressions, CTR, position, age)

**Safe Language:** "The model identifies pages with a higher likelihood of declining impressions based on observed signals. Content teams could use this ranked list to prioritize review candidates."

In [4]:
print("=" * 50)
print("INTENDED USE AND LIMITS")
print("=" * 50)

print("""
Intended Use:
  Who: Content managers, SEO teams, editorial leads
  What: Prioritize pages for review and refresh
  When: Weekly review cycle (recommended)
  How: Review top 50 ranked pages, verify reason codes, take action

Limits:
   Not a guarantee of recovery
   Not causal — based on observed patterns
   Not a replacement for human judgment
   Not for all content types (seasonal content may be misclassified)
   Limited to observed signals (impressions, CTR, position, age)

Safe Language (per writing-honest-claims skill):
  "The model identifies pages with a higher likelihood of declining impressions
  based on observed signals. Content teams could use this ranked list to
  prioritize review candidates."

Banned:
   "The model predicts which pages will decline"
   "Refreshing them will recover their traffic"
""")

INTENDED USE AND LIMITS

Intended Use:
  Who: Content managers, SEO teams, editorial leads
  What: Prioritize pages for review and refresh
  When: Weekly review cycle (recommended)
  How: Review top 50 ranked pages, verify reason codes, take action

Limits:
   Not a guarantee of recovery
   Not causal — based on observed patterns
   Not a replacement for human judgment
   Not for all content types (seasonal content may be misclassified)
   Limited to observed signals (impressions, CTR, position, age)

Safe Language (per writing-honest-claims skill):
  "The model identifies pages with a higher likelihood of declining impressions 
  based on observed signals. Content teams could use this ranked list to 
  prioritize review candidates."

Banned:
   "The model predicts which pages will decline"
   "Refreshing them will recover their traffic"



## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*


**What a Person Must Check:**
1. Seasonality — is low CTR seasonal?
2. Intent match — does page match user intent?
3. Content quality — is content actually thin?
4. Competitor context — are competitors better?
5. Technical issues — indexing or loading problems?

**No-Go List (Never Automate):**
-  Full content rewrite — requires editorial judgment
-  Page deletion — requires business context
-  URL changes — can cause SEO damage
-  Monetization decisions — requires revenue impact
-  Brand-sensitive changes — brand voice needs human oversight

**Review Process:** Top 10 weekly → check seasonality → verify reason codes → prioritize → assign to team.

In [5]:
print("=" * 50)
print("HUMAN REVIEW + NO-GO LIST")
print("=" * 50)

print("""
What a Person Must Check:
  1. Seasonality: Is low CTR due to seasonal demand drop?
  2. Intent match: Does the page match user's search intent?
  3. Content quality: Is the content actually thin?
  4. Competitor context: Are competitors offering better content?
  5. Technical issues: Are there indexing or loading issues?

The No-Go List (What Should NEVER Be Automated):
   Full content rewrite — requires editorial judgment
   Page deletion — requires business context
   URL changes — can cause SEO damage
   Monetization decisions — requires revenue impact assessment
   Brand-sensitive changes — brand voice requires human oversight

Human Review Process:
  1. Review top 10 recommendations weekly
  2. Check for seasonal patterns
  3. Verify reason codes against actual page context
  4. Prioritize based on business goals
  5. Assign to appropriate team member
""")

HUMAN REVIEW + NO-GO LIST

What a Person Must Check:
  1. Seasonality: Is low CTR due to seasonal demand drop?
  2. Intent match: Does the page match user's search intent?
  3. Content quality: Is the content actually thin?
  4. Competitor context: Are competitors offering better content?
  5. Technical issues: Are there indexing or loading issues?

The No-Go List (What Should NEVER Be Automated):
   Full content rewrite — requires editorial judgment
   Page deletion — requires business context
   URL changes — can cause SEO damage
   Monetization decisions — requires revenue impact assessment
   Brand-sensitive changes — brand voice requires human oversight

Human Review Process:
  1. Review top 10 recommendations weekly
  2. Check for seasonal patterns
  3. Verify reason codes against actual page context
  4. Prioritize based on business goals
  5. Assign to appropriate team member



## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Stale Triggers:**
| Trigger | Action |
|---------|--------|
| Precision@50 < 0.65 | Retrain model |
| Page performance shifts | Review features |
| CTR distribution changes | Re-evaluate thresholds |
| New content types added | Re-evaluate model |

**Retraining Plan:**
- Data refresh: Monthly
- Feature engineering: Quarterly
- Model retraining: Quarterly
- Validation: Quarterly
- Playbook update: Annually

**Monitoring Metrics:**
- Precision@50: Target ≥ 0.65, Warning < 0.60
- Base rate: Track changes
- Feature importance: Monitor for shifts

In [6]:
print("=" * 50)
print("MONITORING / RETRAIN TRIGGERS")
print("=" * 50)

print("""
Recommendations Stale Triggers:
   Precision@50 drops below 0.65 → Retrain model
   Page performance shifts significantly → Review features
   CTR distribution changes → Re-evaluate thresholds
   New content types added → Re-evaluate model

Retraining Plan:
  Data refresh: Monthly (pull new warehouse data)
  Feature engineering: Quarterly (review and update)
  Model retraining: Quarterly (retrain with new data)
  Validation: Quarterly (re-run client-holdout split)
  Playbook update: Annually (update action recommendations)

Monitoring Metrics:
  Precision@50: Target ≥ 0.65, Warning < 0.60
  Base rate: Track changes
  Feature importance: Monitor for significant shifts
""")

MONITORING / RETRAIN TRIGGERS

Recommendations Stale Triggers:
   Precision@50 drops below 0.65 → Retrain model
   Page performance shifts significantly → Review features
   CTR distribution changes → Re-evaluate thresholds
   New content types added → Re-evaluate model

Retraining Plan:
  Data refresh: Monthly (pull new warehouse data)
  Feature engineering: Quarterly (review and update)
  Model retraining: Quarterly (retrain with new data)
  Validation: Quarterly (re-run client-holdout split)
  Playbook update: Annually (update action recommendations)

Monitoring Metrics:
  Precision@50: Target ≥ 0.65, Warning < 0.60
  Base rate: Track changes
  Feature importance: Monitor for significant shifts



## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*


**Exported Files:**
| File | Purpose |
|------|---------|
| `work/outputs/action_queue.csv` | Top 100 ranked actions |
| `work/outputs/action_summary.json` | Summary metrics |

**Summary Statistics:**
- Total pages scored: 176,738
- Precision@50: 1.000
- Base rate (test set): 0.396
- Action counts: other (116,668), high_impressions_low_ctr (59,393), position_opportunity (475), stale_visible_page (202)

In [9]:
print("=" * 50)
print("EXPORTS FOR THE PAPER")
print("=" * 50)

import json
import os

# Ensure outputs directory exists
os.makedirs('work/outputs', exist_ok=True)

# Use the existing queue from Section 1
if 'queue' in locals() and len(queue) > 0:
    # Export top 100 for the paper
    top_100 = queue.head(100).copy()
    top_100.to_csv('work/outputs/action_queue.csv', index=False)
    print(f" Exported top 100 actions to work/outputs/action_queue.csv")

    # Export summary statistics
    action_counts = queue['reason_code'].value_counts().to_dict()
    summary = {
        'total_pages': len(queue),
        'action_counts': action_counts,
        'precision_at_50': 1.000,  # From ML-08/ML-09 results
        'base_rate': 0.396,        # From client-holdout test set
        'top_actions': queue.head(10)[['reason_code', 'impressions_90d', 'ctr_90d']].to_dict(orient='records')
    }

    with open('work/outputs/action_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    print(f" Exported summary to work/outputs/action_summary.json")

    print("\nSummary Statistics:")
    print(f"  Total pages scored: {len(queue):,}")
    print(f"  Precision@50: 1.000")
    print(f"  Base rate (test set): 0.396")
    print(f"  Actions by type:")
    for code, count in action_counts.items():
        print(f"    - {code}: {count:,}")
else:
    print(" Queue not available. Please run Section 1 first.")

EXPORTS FOR THE PAPER
 Exported top 100 actions to work/outputs/action_queue.csv
 Exported summary to work/outputs/action_summary.json

Summary Statistics:
  Total pages scored: 176,738
  Precision@50: 1.000
  Base rate (test set): 0.396
  Actions by type:
    - other: 116,668
    - high_impressions_low_ctr: 59,393
    - position_opportunity: 475
    - stale_visible_page: 202


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.